<a href="https://colab.research.google.com/github/kavin43325011-svg/Pak-prayit-semester-2/blob/main/Pratikum_PBO_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **JOBSHEET 11**

In [ ]:
!pip install streamlit pandas -q
!npm install -g localtunnel -q

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [ ]:
%%writefile konfigurasi.py

import os

BASE_DIR = os.getcwd()

NAMA_DB = "pengeluaran_harian.db"
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)

KATEGORI_PENGELUARAN = [
    "Makanan",
    "Transportasi",
    "Hiburan",
    "Tagihan",
    "Belanja",
    "Kesehatan",
    "Pendidikan",
    "Lainnya"
]

KATEGORI_DEFAULT = "Lainnya"

Overwriting konfigurasi.py


In [ ]:
%%writefile database.py

import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection():

    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        return conn

    except sqlite3.Error:
        return None


def execute_query(query, params=None):

    conn = get_db_connection()

    if not conn:
        return None

    try:

        cursor = conn.cursor()

        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)

        conn.commit()

        return cursor.lastrowid

    except sqlite3.Error:

        conn.rollback()
        return None

    finally:

        conn.close()


def fetch_query(query, params=None, fetch_all=True):

    conn = get_db_connection()

    if not conn:
        return None

    try:

        cursor = conn.cursor()

        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)

        if fetch_all:
            return cursor.fetchall()

        return cursor.fetchone()

    finally:

        conn.close()


def get_dataframe(query, params=None):

    conn = get_db_connection()

    if not conn:
        return pd.DataFrame()

    try:

        return pd.read_sql_query(
            query,
            conn,
            params=params
        )

    except:

        return pd.DataFrame()

    finally:

        conn.close()


def setup_database_initial():

    conn = get_db_connection()

    if not conn:
        return False

    try:

        cursor = conn.cursor()

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS transaksi(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL,
            kategori TEXT,
            tanggal DATE NOT NULL
        )
        """)

        conn.commit()

        return True

    except:

        return False

    finally:

        conn.close()

Overwriting database.py


In [ ]:
%%writefile model.py

import datetime

class Transaksi:

    def __init__(
        self,
        deskripsi,
        jumlah,
        kategori,
        tanggal,
        id_transaksi=None
    ):

        self.id = id_transaksi
        self.deskripsi = deskripsi
        self.jumlah = float(jumlah)
        self.kategori = kategori

        if isinstance(tanggal, str):

            self.tanggal = datetime.datetime.strptime(
                tanggal,
                "%Y-%m-%d"
            ).date()

        else:

            self.tanggal = tanggal

    def to_dict(self):

        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

Overwriting model.py


In [ ]:
%%writefile manajer_anggaran.py

import database
import pandas as pd

from model import Transaksi


class AnggaranHarian:

    def __init__(self):

        database.setup_database_initial()

    def tambah_transaksi(self, transaksi):

        sql = """
        INSERT INTO transaksi
        (
            deskripsi,
            jumlah,
            kategori,
            tanggal
        )
        VALUES
        (
            ?,?,?,?
        )
        """

        params = (
            transaksi.deskripsi,
            transaksi.jumlah,
            transaksi.kategori,
            transaksi.tanggal.strftime("%Y-%m-%d")
        )

        return database.execute_query(
            sql,
            params
        )

    def get_dataframe_transaksi(self):

        sql = """
        SELECT
            id,
            tanggal,
            kategori,
            deskripsi,
            jumlah
        FROM transaksi
        ORDER BY id DESC
        """

        return database.get_dataframe(sql)

    def hitung_total_pengeluaran(self):

        sql = """
        SELECT SUM(jumlah)
        FROM transaksi
        """

        hasil = database.fetch_query(
            sql,
            fetch_all=False
        )

        if hasil and hasil[0]:
            return hasil[0]

        return 0

    def get_pengeluaran_per_kategori(self):

        sql = """
        SELECT
            kategori,
            SUM(jumlah) as total
        FROM transaksi
        GROUP BY kategori
        """

        rows = database.fetch_query(sql)

        data = {}

        if rows:

            for row in rows:

                data[row["kategori"]] = row["total"]

        return data

    def hapus_transaksi(self, id_transaksi):

        conn = database.get_db_connection()

        cursor = conn.cursor()

        cursor.execute(
            "DELETE FROM transaksi WHERE id=?",
            (id_transaksi,)
        )

        conn.commit()

        berhasil = cursor.rowcount > 0

        conn.close()

        return berhasil

Overwriting manajer_anggaran.py


In [ ]:
%%writefile streamlit_app.py

import streamlit as st
import pandas as pd
import datetime

from model import Transaksi
from manajer_anggaran import AnggaranHarian
from konfigurasi import KATEGORI_PENGELUARAN

# ======================
# KONFIGURASI
# ======================

st.set_page_config(
    page_title="Catatan Pengeluaran",
    page_icon="💰",
    layout="wide"
)

anggaran = AnggaranHarian()

# ======================
# FORMAT RUPIAH
# ======================

def format_rp(angka):

    return f"Rp {angka:,.0f}".replace(",", ".")

# ======================
# HALAMAN INPUT
# ======================

def halaman_input():

    st.title("💸 Tambah Pengeluaran")

    with st.form("form_transaksi"):

        deskripsi = st.text_input(
            "Deskripsi"
        )

        kategori = st.selectbox(
            "Kategori",
            KATEGORI_PENGELUARAN
        )

        jumlah = st.number_input(
            "Jumlah (Rp)",
            min_value=1.0,
            step=1000.0
        )

        tanggal = st.date_input(
            "Tanggal",
            value=datetime.date.today()
        )

        submit = st.form_submit_button(
            "Simpan"
        )

        if submit:

            transaksi = Transaksi(
                deskripsi,
                jumlah,
                kategori,
                tanggal
            )

            hasil = anggaran.tambah_transaksi(
                transaksi
            )

            if hasil:

                st.success(
                    "Transaksi berhasil disimpan"
                )

            else:

                st.error(
                    "Gagal menyimpan transaksi"
                )

# ======================
# HALAMAN RIWAYAT
# ======================

def halaman_riwayat():

    st.title("📋 Riwayat Pengeluaran")

    df = anggaran.get_dataframe_transaksi()

    if df.empty:

        st.info(
            "Belum ada transaksi."
        )

    else:

        st.dataframe(
            df,
            use_container_width=True,
            hide_index=True
        )

    st.divider()

    st.subheader(
        "🗑 Hapus Transaksi"
    )

    id_hapus = st.number_input(
        "Masukkan ID Transaksi",
        min_value=1,
        step=1
    )

    if st.button(
        "Hapus Data"
    ):

        st.session_state["konfirmasi"] = True
        st.session_state["id_hapus"] = id_hapus

    if st.session_state.get(
        "konfirmasi",
        False
    ):

        st.warning(
            f"Yakin ingin menghapus transaksi ID {st.session_state['id_hapus']} ?"
        )

        col1, col2 = st.columns(2)

        with col1:

            if st.button(
                "Ya, Hapus"
            ):

                sukses = anggaran.hapus_transaksi(
                    st.session_state["id_hapus"]
                )

                if sukses:

                    st.success(
                        "Data berhasil dihapus"
                    )

                    st.session_state[
                        "konfirmasi"
                    ] = False

                    st.rerun()

                else:

                    st.error(
                        "ID tidak ditemukan"
                    )

        with col2:

            if st.button(
                "Batal"
            ):

                st.session_state[
                    "konfirmasi"
                ] = False

                st.rerun()

# ======================
# HALAMAN RINGKASAN
# ======================

def halaman_ringkasan():

    st.title("📊 Ringkasan Pengeluaran")

    total = anggaran.hitung_total_pengeluaran()

    st.metric(
        "Total Pengeluaran",
        format_rp(total)
    )

    st.divider()

    data = anggaran.get_pengeluaran_per_kategori()

    if not data:

        st.info(
            "Belum ada data."
        )

    else:

        df = pd.DataFrame(
            data.items(),
            columns=[
                "Kategori",
                "Total"
            ]
        )

        df["Total"] = df["Total"].astype(
            float
        )

        col1, col2 = st.columns(2)

        with col1:

            st.subheader(
                "Tabel Kategori"
            )

            st.dataframe(
                df,
                use_container_width=True,
                hide_index=True
            )

        with col2:

            st.subheader(
                "Grafik Kategori"
            )

            st.bar_chart(
                df.set_index(
                    "Kategori"
                )["Total"]
            )

# ======================
# MENU SIDEBAR
# ======================

st.sidebar.title(
    "💰 Catatan Pengeluaran"
)

menu = st.sidebar.radio(
    "Pilih Menu",
    [
        "Tambah",
        "Riwayat",
        "Ringkasan"
    ]
)

if menu == "Tambah":

    halaman_input()

elif menu == "Riwayat":

    halaman_riwayat()

elif menu == "Ringkasan":

    halaman_ringkasan()

st.sidebar.markdown("---")
st.sidebar.info(
    "Jobsheet 11 OOP"
)

Writing streamlit_app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!wget -q -O - ipv4.icanhazip.com

34.123.91.74


In [ ]:
!lt --port 8501

your url is: https://free-jars-brush.loca.lt
